In [ ]:
# --- Data wrangling ---
import pandas as pd
import numpy as np

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Utilities ---
import os
import time
import warnings
import joblib

warnings.filterwarnings("ignore")

# --- ML: core ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,   
    f1_score,
    roc_auc_score,
)

# --- Imbalance handling ---
from imblearn.pipeline import Pipeline as ImbPipeline   
from imblearn.over_sampling import SMOTE

# --- Main model ---
import xgboost as xgb

# ── Version check ─────────────────────────────────────────────────────────────
# early_stopping_rounds moved to the XGBoost constructor in v2.0.
# If you get unexpected errors, run: pip install --upgrade xgboost
xgb_major = int(xgb.__version__.split(".")[0])
assert xgb_major >= 2, (
    f"❌ XGBoost {xgb.__version__} detected. This file requires XGBoost ≥ 2.0.\n"
    "   Fix: pip install --upgrade xgboost"
)

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── File paths ────────────────────────────────────────────────────────────────
PARQUET_TRAIN = "data/train.parquet"   # INPUT  (Stage 6 output)
PARQUET_TEST  = "data/test.parquet"    # INPUT  (Stage 6 output)
MODELS_DIR    = "models"               # OUTPUT directory

# ── Feature list (must match Stage 5's FEATURE_COLS exactly) ─────────────────
# This is the canonical ordered list. Column order matters: XGBoost sees
# a plain array of numbers — position 0, position 1, ... — not column names.
# A single swap silently corrupts every prediction without raising an error.
FEATURE_COLS = [
    "amount",
    "type_TRANSFER",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "errorBalanceOrig",
    "errorBalanceDest",
    "flag_orig_zero_after",
    "flag_dest_zero_before",
    "flag_dest_zero_both",
]
TARGET_COL = "isFraud"

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FRAUD_PALETTE = {0: "#2196F3", 1: "#F44336"}   # Blue = legit, Red = fraud

print(f"✅ Imports complete.  XGBoost {xgb.__version__} detected.")
print(f"   FEATURE_COLS: {FEATURE_COLS}")

## Class Imbalance

In [ ]:
for path in [PARQUET_TRAIN, PARQUET_TEST]:
    assert os.path.exists(path), (
        f"\n '{path}' not found.\n"
        "Run all cells in fraud_detection_stage5_6.ipynb first.\n"
        "Cell 18 there saves both train.parquet and test.parquet."
    )

train = pd.read_parquet(PARQUET_TRAIN)
test  = pd.read_parquet(PARQUET_TEST)

X_train = train[FEATURE_COLS]
y_train = train[TARGET_COL]
X_test  = test[FEATURE_COLS]
y_test  = test[TARGET_COL]

# Recompute the imbalance ratio from training labels only
n_neg            = int((y_train == 0).sum())
n_pos            = int((y_train == 1).sum())
SCALE_POS_WEIGHT = round(n_neg / n_pos, 2)

print(f" Splits loaded.")
print(f"   X_train : {X_train.shape[0]:>9,} rows × {X_train.shape[1]} features")
print(f"   X_test  : {X_test.shape[0]:>9,} rows × {X_test.shape[1]} features")
print(f"   Train fraud rate : {y_train.mean()*100:.4f}%  ({n_pos:,} frauds)")
print(f"   Test  fraud rate : {y_test.mean()*100:.4f}%  ({y_test.sum():,} frauds)")
print(f"   SCALE_POS_WEIGHT : {SCALE_POS_WEIGHT}")
print()
print("  X_test / y_test are now locked. They will NOT be touched until Stage 9.")

In [ ]:
print("Class ratio in training data:")
print(f"  Legitimate : {n_neg:>9,}  ({n_neg/(n_neg+n_pos)*100:.3f}%)")
print(f"  Fraud      : {n_pos:>9,}  ({n_pos/(n_neg+n_pos)*100:.4f}%)")
print(f"  Ratio      :  {SCALE_POS_WEIGHT:.0f}:1  (legitimate per fraud)")
print()
print("Effect on a naive model:")
print(f"  Predicting 'never fraud' achieves {n_neg/(n_neg+n_pos)*100:.3f}% accuracy.")
print("  That model catches zero fraud. Accuracy is the wrong metric here.")
print()
print("Two solutions:")
print("  A. Cost reweighting  → scale_pos_weight / class_weight='balanced'")
print("  B. Data rebalancing  → SMOTE oversampling")

In [ ]:
print(f"SCALE_POS_WEIGHT = {n_neg:,} / {n_pos:,} = {SCALE_POS_WEIGHT}")
print()
print("This tells XGBoost:")
print(f"  'Every missed fraud costs the model {SCALE_POS_WEIGHT:.0f}× more")
print("   than every missed legitimate transaction during training.'")
print()
print("XGBoost usage in Cell 12:")
print(f"  xgb.XGBClassifier(scale_pos_weight={SCALE_POS_WEIGHT}, ...)")
print()
print("LR equivalent (for the baseline in Cell 11):")
print("  LogisticRegression(class_weight='balanced', ...)")

In [ ]:
print("SMOTE key parameters:")
print()
print("  sampling_strategy : target minority:majority ratio after resampling")
print("                      0.1 → oversample fraud until 1:10  (not 1:1)")
print("                      'auto' → oversample to 1:1 (often too aggressive)")
print()
print("  k_neighbors       : how many fraud neighbours to consider per point")
print("                      default=5; increase if fraud points are sparse")
print()
print("  random_state      : must match RANDOM_STATE for reproducibility")
print()
print("Our choice → sampling_strategy=0.1 (explained in Cell 7)")

In [ ]:
print("Pipeline pattern (used in Cells 11 and 13):")
print()
print("  from imblearn.pipeline import Pipeline as ImbPipeline")
print()
print("  Model A — LR Baseline:")
print("  ImbPipeline([")
print('    ("scaler", StandardScaler()),')
print('    ("model",  LogisticRegression(class_weight="balanced", ...))')
print("  ])")
print()
print("  Model C — XGB + SMOTE:")
print("  ImbPipeline([")
print('    ("smote", SMOTE(sampling_strategy=0.1, ...)),')
print('    ("model", XGBClassifier(scale_pos_weight=1, ...))')
print("  ])")
print()
print("  ← Note: scale_pos_weight=1 when SMOTE handles the imbalance.")

In [ ]:
print("sampling_strategy comparison:")
print()
print(f"  Raw data                   : 1 fraud : {SCALE_POS_WEIGHT:.0f} legitimate")
print(f"  SMOTE @ strategy='auto'    : 1 fraud : 1  legitimate  (50/50)")
print(f"  SMOTE @ strategy=0.1       : 1 fraud : 10 legitimate  (our choice)")
print()
print("  At strategy=0.1:")
print(f"    Real fraud rows in train  : {n_pos:,}")
fraud_target = int(n_neg * 0.1)
synth_needed = max(0, fraud_target - n_pos)
print(f"    Target fraud after SMOTE  : {fraud_target:,}")
print(f"    Synthetic rows created    : ~{synth_needed:,}")
print(f"    Total train rows after    : ~{n_neg + fraud_target:,}")
print()
print("  Tradeoff: less extreme resampling → better probability calibration")
print("            but model still needs threshold tuning (Stage 10).")

In [ ]:
VAL_FRAC    = 0.10
val_cut_idx = int(len(X_train) * (1 - VAL_FRAC))

X_tr  = X_train.iloc[:val_cut_idx]
y_tr  = y_train.iloc[:val_cut_idx]
X_val = X_train.iloc[val_cut_idx:]
y_val = y_train.iloc[val_cut_idx:]

print(f"Validation split (temporal, last {VAL_FRAC*100:.0f}% of training rows):")
print(f"  X_tr  (XGBoost fits on this)   : {X_tr.shape[0]:>9,} rows")
print(f"  X_val (early stopping watches)  : {X_val.shape[0]:>9,} rows")
print(f"  X_test (Stage 9 evaluation)     : {X_test.shape[0]:>9,} rows  ← untouched")
print()
print(f"  Val fraud rate : {y_val.mean()*100:.4f}%  ({y_val.sum():,} frauds)")
print()
print("✅ Three-way data partition established.")

In [ ]:
print("=" * 62)
print("  STAGE 7 COMPLETE — IMBALANCE STRATEGY SUMMARY")
print("=" * 62)
print()
print("  Three model architectures to train in Stage 8:")
print()
print("  Model A — Logistic Regression Baseline")
print("    ImbPipeline([StandardScaler, LR(class_weight='balanced')])")
print("    Fits on: full X_train")
print("    Purpose: establishes the linear floor; any model that barely")
print("             beats this has weak non-linear signal or a bug.")
print()
print("  Model B — XGBoost + scale_pos_weight  ← PRIMARY MODEL")
print(f"    XGBClassifier(scale_pos_weight={SCALE_POS_WEIGHT}, early_stopping)")
print("    Fits on: X_tr (90% of train); early-stops on X_val")
print("    Purpose: the main portfolio model; simpler and often stronger.")
print()
print("  Model C — XGBoost + SMOTE Pipeline    ← COMPARISON")
print("    ImbPipeline([SMOTE(strategy=0.1), XGBClassifier()])")
print("    Fits on: full X_train")
print("    Purpose: demonstrates the correct Pipeline pattern; compared")
print("             against Model B via PR-AUC in Stage 9.")
print()
print("  Primary metric for comparison: PR-AUC (Average Precision)")
print("  Secondary metrics: Recall, F1, Confusion Matrix")
print()
print("  NEXT: Stage 8 — Train all three models")

## Model Training

In [ ]:
print("XGBoost hyperparameter reference printed. Proceed to Cell 11 to train.")
print()
print("  n_estimators=1000   → ceiling; early stopping finds the real number")
print("  learning_rate=0.05  → careful learning pace")
print("  max_depth=6         → captures feature interactions without overfitting")
print("  subsample=0.8       → row-level randomness, fights overfitting")
print("  colsample_bytree=0.8→ feature-level randomness")
print("  eval_metric='aucpr' → early stopping watches PR-AUC directly")
print("  early_stopping_rounds=50")
print("  tree_method='hist'  → fastest CPU algorithm")

In [ ]:
print("Training Model A: Logistic Regression Baseline...")
print("  (with StandardScaler + class_weight='balanced')")
print()

pipe_lr = ImbPipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(
        class_weight  = "balanced",
        solver        = "saga",
        max_iter      = 500,
        C             = 1.0,       # inverse regularisation strength; 1.0 = default
        random_state  = RANDOM_STATE,
        n_jobs        = -1,        # use all CPU cores
    ))
])

t0 = time.time()
pipe_lr.fit(X_train, y_train)
lr_time = time.time() - t0

print(f"✅ LR baseline trained in {lr_time:.1f}s")
print()

# Quick sanity: confirm it's not collapsing to all-zero predictions
lr_val_preds = pipe_lr.predict(X_val)
lr_val_fraud_predicted = lr_val_preds.sum()
print(f"  Val fraud predictions  : {lr_val_fraud_predicted:,}  (0 = model is broken)")
print(f"  Val actual frauds      : {y_val.sum():,}")

if lr_val_fraud_predicted == 0:
    print("    Model predicts NO fraud — check class_weight or C parameter.")
else:
    print("   Model is predicting some fraud — sanity check passed.")


In [ ]:
print("Training Model B: XGBoost + scale_pos_weight (primary model)...")
print(f"  scale_pos_weight = {SCALE_POS_WEIGHT}")
print(f"  Fitting on X_tr ({len(X_tr):,} rows), validating on X_val ({len(X_val):,} rows)")
print(f"  Early stopping: 50 rounds without PR-AUC improvement")
print()

model_xgb_spw = xgb.XGBClassifier(
    n_estimators          = 1000,
    learning_rate         = 0.05,
    max_depth             = 6,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = SCALE_POS_WEIGHT,
    eval_metric           = "aucpr",         # PR-AUC: our headline metric
    early_stopping_rounds = 50,
    tree_method           = "hist",          # fastest CPU algorithm
    random_state          = RANDOM_STATE,
    n_jobs                = -1,
    verbosity             = 0,               # suppress XGBoost system messages
)

t0 = time.time()
model_xgb_spw.fit(
    X_tr, y_tr,
    eval_set  = [(X_val, y_val)],
    verbose   = 100,
)
xgb_spw_time = time.time() - t0

print()
print(f" XGBoost (scale_pos_weight) trained in {xgb_spw_time:.1f}s")
print(f"   Best iteration : {model_xgb_spw.best_iteration}")
print(f"   Best val AUCPR : {model_xgb_spw.best_score:.4f}")

In [ ]:
print("Training Model C: XGBoost + SMOTE Pipeline (comparison model)...")
print(f"  SMOTE sampling_strategy=0.1  (fraud:legit ratio → ~1:10)")
print(f"  n_estimators=300 (fixed; no early stopping with SMOTE pipeline)")
print(f"  Fitting on full X_train ({len(X_train):,} rows)")
print()

pipe_xgb_smote = ImbPipeline([
    ("smote", SMOTE(
        sampling_strategy = 0.1,
        k_neighbors       = 5,
        random_state      = RANDOM_STATE,
        # n_jobs removed — not supported in current imbalanced-learn
    )),
    ("model", xgb.XGBClassifier(
        n_estimators     = 300,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = 1,
        eval_metric      = "aucpr",
        tree_method      = "hist",
        random_state     = RANDOM_STATE,
        n_jobs           = -1,        # n_jobs is valid on XGBClassifier
        verbosity        = 0,
    )),
])

t0 = time.time()
pipe_xgb_smote.fit(X_train, y_train)
xgb_smote_time = time.time() - t0

print(f" XGBoost + SMOTE pipeline trained in {xgb_smote_time:.1f}s")
print()

# SMOTE timing note: the extra time vs Model B is almost entirely SMOTE's
# synthesis step — creating synthetic fraud rows via KNN interpolation.
# The actual XGBoost fit on the resampled data may be slightly faster
# (fewer rounds, no early stopping) but SMOTE's overhead is noticeable.
print(f"  Training time comparison:")
print(f"    LR baseline         : {lr_time:6.1f}s")
print(f"    XGB (scale_pos_wt)  : {xgb_spw_time:6.1f}s  (incl. early stopping)")
print(f"    XGB + SMOTE pipeline: {xgb_smote_time:6.1f}s  (incl. SMOTE synthesis)")

In [ ]:
print("Validation PR-AUC preview (X_val — 10% of training data):")
print()
print("  This is a quick sanity check, NOT the final evaluation.")
print("  Full PR curves + confusion matrices + threshold tuning → Stage 9-10.")
print()

results_preview = []

for name, model in [
    ("LR Baseline",             pipe_lr),
    ("XGB + scale_pos_weight",  model_xgb_spw),
    ("XGB + SMOTE Pipeline",    pipe_xgb_smote),
]:
    y_prob = model.predict_proba(X_val)[:, 1]
    pr_auc = average_precision_score(y_val, y_prob)
    roc    = roc_auc_score(y_val, y_prob)
    results_preview.append({"Model": name, "Val PR-AUC": pr_auc, "Val ROC-AUC": roc})

df_preview = pd.DataFrame(results_preview).sort_values("Val PR-AUC", ascending=False)
df_preview[["Val PR-AUC", "Val ROC-AUC"]] = df_preview[["Val PR-AUC", "Val ROC-AUC"]].round(4)
df_preview = df_preview.reset_index(drop=True)
df_preview.index += 1

print(df_preview.to_string())
print()

# ── Visual: PR-AUC bar chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#F44336", "#2196F3", "#4CAF50"]   # red, blue, green
bars = ax.barh(
    df_preview["Model"],
    df_preview["Val PR-AUC"],
    color=colors[:len(df_preview)],
    alpha=0.75,
    height=0.5,
)
for bar, val in zip(bars, df_preview["Val PR-AUC"]):
    ax.text(
        bar.get_width() - 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", ha="right", color="white", fontweight="bold"
    )
ax.set_xlabel("Validation PR-AUC (higher is better)")
ax.set_title("Model Comparison — Validation PR-AUC Preview", fontweight="bold")
ax.axvline(y_val.mean(), color="black", linestyle="--", linewidth=1,
           label=f"Random baseline ({y_val.mean():.4f})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"  Random baseline (fraud rate) : {y_val.mean():.4f}")
print("  Any model above this line has meaningful discriminating power.")

In [ ]:
best_model_name = df_preview.iloc[0]["Model"]
best_prauc      = df_preview.iloc[0]["Val PR-AUC"]

print("Model selection framework:")
print()
print(f"  Preliminary preferred model : {best_model_name}")
print(f"  Validation PR-AUC           : {best_prauc:.4f}")
print()
print("  All three models will be:")
print("    ✓ Saved to models/ in Cell 16")
print("    ✓ Fully evaluated on X_test in Stage 9")
print("    ✓ Compared by PR curve, confusion matrix, and threshold analysis")
print()
print("  Elimination rules for Stage 9:")
print("    If LR PR-AUC ≥ XGBoost PR-AUC → investigate for a bug or leakage")
print("    If SMOTE PR-AUC > SPW by > 1%  → use SMOTE as primary")
print("    If SPW PR-AUC ≥ SMOTE          → prefer SPW (simpler)")
print("    If all models ≈ equal           → features are the main driver")

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

paths = {
    "lr_baseline" : os.path.join(MODELS_DIR, "lr_baseline.joblib"),
    "xgb_spw"     : os.path.join(MODELS_DIR, "xgb_spw.joblib"),
    "xgb_smote"   : os.path.join(MODELS_DIR, "xgb_smote.joblib"),
    "model_meta"  : os.path.join(MODELS_DIR, "model_meta.joblib"),
}

joblib.dump(pipe_lr,        paths["lr_baseline"])
joblib.dump(model_xgb_spw,  paths["xgb_spw"])
joblib.dump(pipe_xgb_smote, paths["xgb_smote"])

model_meta = {
    "feature_cols"        : FEATURE_COLS,
    "target_col"          : TARGET_COL,
    "scale_pos_weight"    : SCALE_POS_WEIGHT,
    "random_state"        : RANDOM_STATE,
    "xgb_best_iteration"  : model_xgb_spw.best_iteration,
    "xgb_best_val_aucpr"  : round(model_xgb_spw.best_score, 6),
    "val_preview"         : df_preview.to_dict("records"),
    # threshold will be added by Stage 10 after tuning
}
joblib.dump(model_meta, paths["model_meta"])

# Print file sizes for awareness
print("Models saved:\n")
for label, path in paths.items():
    size_mb = os.path.getsize(path) / 1024**2
    print(f"  {label:<16} → {path}  ({size_mb:.2f} MB)")

print()
print("How to reload in Stage 9:")
print("  import joblib")
print("  pipe_lr        = joblib.load('models/lr_baseline.joblib')")
print("  model_xgb_spw  = joblib.load('models/xgb_spw.joblib')")
print("  pipe_xgb_smote = joblib.load('models/xgb_smote.joblib')")
print("  meta           = joblib.load('models/model_meta.joblib')")
print("  FEATURE_COLS   = meta['feature_cols']   # exact order guaranteed")

In [ ]:
print("=" * 62)
print("  STAGE 8 COMPLETE — MODEL TRAINING SUMMARY")
print("=" * 62)
print()
print(f"  Models trained  : 3")
print()
print(f"  A — LR Baseline")
print(f"      Architecture  : ImbPipeline([StandardScaler, LogisticRegression])")
print(f"      Trained on    : full X_train ({len(X_train):,} rows)")
print(f"      Time          : {lr_time:.1f}s")
print()
print(f"  B — XGBoost + scale_pos_weight  (PRIMARY)")
print(f"      Architecture  : XGBClassifier(scale_pos_weight={SCALE_POS_WEIGHT})")
print(f"      Trained on    : X_tr ({len(X_tr):,} rows)")
print(f"      Best iteration: {model_xgb_spw.best_iteration} / 1000")
print(f"      Best val AUCPR: {model_xgb_spw.best_score:.4f}")
print(f"      Time          : {xgb_spw_time:.1f}s")
print()
print(f"  C — XGBoost + SMOTE Pipeline (COMPARISON)")
print(f"      Architecture  : ImbPipeline([SMOTE(0.1), XGBClassifier])")
print(f"      Trained on    : full X_train ({len(X_train):,} rows)")
print(f"      n_estimators  : 300 (fixed)")
print(f"      Time          : {xgb_smote_time:.1f}s")
print()
print("  Saved to models/:")
for label, path in paths.items():
    print(f"     {os.path.basename(path)}")
print()
print("  NEXT STAGES:")
print("  ┌──────────────────────────────────────────────────────┐")
print("  │  Stage 9  — Full evaluation on X_test               │")
print("  │             PR curves, confusion matrices, F1        │")
print("  │             → picks the primary model                │")
print("  │                                                       │")
print("  │  Stage 10 — Threshold tuning                         │")
print("  │             PR curve + business cost framing         │")
print("  │             → sets the decision threshold            │")
print("  │                                                       │")
print("  │  Stage 11 — SHAP explainability                      │")
print("  │             TreeSHAP on the primary XGBoost model    │")
print("  │             → local + global feature attributions    │")
print("  └──────────────────────────────────────────────────────┘")